# Day 4b. Build it, then prove it

A complete mini-system, end to end: **IT-helpdesk triage**, classify, route,
draft, and then the part most people skip: **an eval that proves it works**.

This is the smallest honest version of the production systems in today's deck.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

## 1 · Design first

Day 2's differentiation, applied: classification is **one typed step inside a
workflow you control**, no tools, no planning, no loop. So it is the **direct
form** (`with_structured_output`), not an agent.

The rule: *start with the direct call; upgrade to the agent when the path needs
planning.*

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Triage(BaseModel):
    """Classification of one IT support ticket."""
    category: Literal["password", "vpn", "software", "hardware", "security", "other"]
    confidence: int = Field(ge=0, le=100, description="How sure you are, 0-100")
    summary: str = Field(description="One line: what the user needs")

classifier = llm.with_structured_output(Triage)      # the direct form

def classify(ticket, extra_instructions=""):
    return classifier.invoke(
        f"Classify this IT support ticket. {extra_instructions}\n\nTicket: {ticket}")

TICKETS = [
    "Locked out of my account again, password reset link never arrives.",
    "VPN drops every 20 minutes when I'm on the dorm wifi.",
    "Need a Tableau licence for my thesis project.",
    "Suspicious login alert from a device in another country?!",
    "My laptop screen flickers when I open the lid.",
    "Everything is slow since Monday. Everything.",
    "Someone sent a link to 'verify my student account'. I clicked it.",
    "How do I connect the beamer in room C01 to HDMI?",
]

classify(TICKETS[0])

In [ ]:
# routing is CODE, the threshold and the security override are decisions you sign
AUTO_THRESHOLD = 85

def route(t: Triage) -> str:
    if t.category == "security":
        return "HUMAN (security desk)"       # always, no threshold applies
    if t.confidence >= AUTO_THRESHOLD and t.category in ("password", "vpn", "software"):
        return "auto-reply"
    return "HUMAN (helpdesk)"

def draft_reply(ticket: str, t: Triage) -> str:
    return llm.invoke(
        f"Draft a 2-sentence friendly IT reply for this {t.category} issue. "
        f"Give the single most likely fix. Ticket: {ticket}").content

results = []
for ticket in TICKETS:
    t = classify(ticket)
    r = route(t)
    results.append((ticket, t, r))
    print(f"{t.category:9} {t.confidence:3}% → {r:22} | {t.summary[:48]}")

**Discuss:** a security incident misfiled as a password reset is not an accuracy
problem, it is a **cost asymmetry** problem. A wrong password reply wastes a
minute; a wrong security reply loses an account.

That is why the security override is unconditional **code**, not a confidence
score. The model classifies; your code decides what is safe to automate.

## 2 · Evals, prove it, don't feel it

"It seemed fine when I tried it" is not an engineering claim. An eval is a test
suite for behavior: **real past inputs + what *right* looked like.**

In [ ]:
EVAL_CASES = [
    ("Locked out of my account again, password reset link never arrives.", "password"),
    ("VPN drops every 20 minutes when I'm on the dorm wifi.",              "vpn"),
    ("Suspicious login alert from a device in another country?!",          "security"),
    ("Someone sent a link to 'verify my student account'. I clicked it.", "security"),
    ("Need a Tableau licence for my thesis project.",                      "software"),
    ("My laptop screen flickers when I open the lid.",                     "hardware"),
]

def run_eval(extra_instructions=""):
    passed, failures = 0, []
    for ticket, expected in EVAL_CASES:
        t = classify(ticket, extra_instructions)
        ok = t.category == expected
        # the check that actually matters: security must NEVER reach auto-reply
        if expected == "security" and route(t) == "auto-reply":
            ok = False
        passed += ok
        if not ok:
            failures.append((ticket[:44], expected, t.category, route(t)))
    print(f"pass rate: {passed}/{len(EVAL_CASES)}")
    for f in failures:
        print("  FAIL:", f)
    return failures

failures = run_eval()

In [ ]:
# found a failure? fix the PROMPT, not the labels, then run it again
run_eval("Anything involving suspicious logins, phishing links, or clicked "
         "links is category 'security', regardless of what else it mentions.")

That loop, **run · read failures · fix · run again**, is the whole discipline.
The eval table is your acceptance test: for a client, a thesis committee, or
your own team.

**Optional. LLM as judge.** Hard checks cannot score a *drafted reply's*
quality. A second structured call can (direct form again: no tools, no loop).

In [ ]:
class Verdict(BaseModel):
    """Judgement of one drafted reply."""
    helpful: bool
    safe: bool = Field(description="No credentials requested, no links, "
                                   "no promises IT can't keep")
    reason: str

judge = llm.with_structured_output(Verdict)

ticket, t, _ = results[0]
draft = draft_reply(ticket, t)
print(draft, "\n")
judge.invoke(f"Judge this IT support draft.\nTicket: {ticket}\nDraft: {draft}")

## 3 · Your turn

**(1)** Add two **adversarial** cases the current prompt gets wrong. Mixed
tickets are the best hunting ground: *"VPN keeps dropping AND I got a weird
login alert."* Fix the prompt until green, **without breaking the other
cases.** (That last part is why you keep the whole suite.)

**(2)** Add a `Verdict`-based check to `run_eval` for the auto-reply drafts: a
run fails if any auto-reply is judged unsafe.

**(3)** Your take-home agent: write five eval cases for it, now, while the
system is fresh in your head. Five minutes of eval beats five weeks of "seems
fine".

In [ ]:
# your adversarial cases here


---
## Wrap, the week

`create_agent` at the center · messages, tools, structured output · middleware,
approvals, skills, backends · MCP, subagents · and an eval that proves any of it
works.

**Take-home** (deck has the worksheet): one task you repeat weekly → inputs →
steps → tools → schema → guardrails → five eval cases → workflow or agent?
Argue the last one.

*Friday: office hours. Bring what you built, especially the broken parts, and
bring a trace.*